## 06 · Build a RAG Pipeline：跑通 Naive RAG

Naive RAG（最小 RAG）只有三步：**检索 → 拼提示词 → 生成答案**。前面每个动作都已经单独看过，现在把它们连起来。

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None

import os
import re
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
if not EMBEDDING_MODEL:
    raise RuntimeError("请在 .env 中配置 EMBEDDING_MODEL。")
model = SentenceTransformer(EMBEDDING_MODEL)

def split_markdown(text, max_chars=800):
    sections = re.split(r"\n(?=#{1,3}\s)", text)
    chunks, current = [], ""
    for section in sections:
        if current and len(current) + len(section) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += section + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for text in split_markdown(path.read_text(encoding="utf-8")):
            items.append({"source": str(path.relative_to(data_dir)), "text": text})
    return items

def build_index(chunks):
    vectors = model.encode([item["text"] for item in chunks], normalize_embeddings=True)
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="fashion_knowledge",
        vectors_config=models.VectorParams(size=vectors.shape[1], distance=models.Distance.COSINE),
    )
    client.upload_points(
        collection_name="fashion_knowledge",
        points=[models.PointStruct(id=i, vector=vector.tolist(), payload=chunk) for i, (vector, chunk) in enumerate(zip(vectors, chunks))],
    )
    return client

In [ ]:
chunks = load_chunks()
qdrant = build_index(chunks)

def search(question, top_k=4):
    question_vector = model.encode(question, normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name="fashion_knowledge", query=question_vector, limit=top_k
    ).points
    return [{**hit.payload, "score": hit.score} for hit in hits]

## 先看检索结果和最终提示词

In [ ]:
question = "SKU-YG301 瑜伽裤的面料成分和防透光要求是什么？"
results = search(question, top_k=4)
context = "\n\n".join(f"[来源：{item['source']}]\n{item['text']}" for item in results)
prompt = f"""请只根据下面的资料回答问题。
资料没有答案时，请回答“现有资料无法回答”，不要猜测。

资料：
{context}

问题：{question}"""
print(prompt[:2500])

## 生成答案

In [ ]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "你是服饰箱包知识库助手，只根据资料回答，并指出依据的来源。"},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print("未调用模型：请配置 API 后重新运行")

到这里，最小 RAG 已经完成。它没有重排、评估或复杂模块，先把核心闭环跑通。